# Jev, an LLM, and confidence-based escalation

**Selected failure category:** Failure to interpret indirect or underspecified requests.

**Binary question:** Did the router fail to interpret the indirect or underspecified scheduling request?

`TRUE` means this particular failure is established under the original routing policy.
`FALSE` means it is not established; the route can still contain another kind of error.
Both judges use the same policy and six worked examples from [the original LLM-only notebook](llm_as_judge.ipynb).

This additional experiment compares Jev, an independent GPT-4.1-mini judge, and a
Jev → LLM cascade. The original notebook retains its GPT-5 / GPT-6 Astra comparison.
Start with its **21 calibration cases: 9 TRUE, 12 FALSE**. The 150 validation cases
remain reserved until you explicitly enable validation after freezing the prompt.


## 1. Setup

Run the cells in order. Use a TypeSafe API key for Jev and an OpenAI key for the LLM.
Keys come from `.env`, Colab Secrets, or hidden input. `auto` reuses a completed
local run; a fresh checkout makes API calls. [Provider and replay options](../README.md).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Colab needs a checkout; locally, open this notebook inside the repository.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    checkout = Path("/content/routing-evals-lab")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "codex/jev-routing-judge",
                        "https://github.com/The-Gen-Academy/Mastering-Agentic-AI-Week4-Session1.git",
                        str(checkout)], check=True)
    os.chdir(checkout)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pandas", "python-dotenv"], check=True)

folders = [Path.cwd(), Path("routing_agent_evals"), Path("AI-Evals-Principles/routing_agent_evals")]
LAB = next(folder.resolve() for folder in folders if (folder / "judge_lab.py").exists())
os.chdir(LAB)
sys.path.insert(0, str(LAB))

In [ ]:
import pandas as pd
from IPython.display import display
from prompts.llm_as_judge_prompt import (
    FAILURE_CATEGORY, FAILURE_QUESTION, CHOICE_CRITERIA,
    JEV_JUDGE_PROMPT, INDIRECT_OR_UNDERSPECIFIED_FAILURE_PROMPT,
)
from judge_lab import (
    load_dataset, request_for, batch, needs_llm, run_cascade,
    comparison_metrics, always_false, confidence_bands, resolved,
)
from notebook_utils import setup_run, save_run

# Freeze the prompt and 0.70 cutoff before enabling the reserved validation set.
# After inspecting validation and making further changes, use a new held-out set.
RUN_VALIDATION = False
SPLIT = "validation" if RUN_VALIDATION else "calibration"

# After editing the prompt file, restart the kernel and run all cells.
client, RUN, manifest = setup_run(LAB, mode="auto", split=SPLIT, prompts={
    "jev": JEV_JUDGE_PROMPT,
    "llm": INDIRECT_OR_UNDERSPECIFIED_FAILURE_PROMPT,
})


## 2. Inspect the question and model input

Both judges receive only `USER_INPUT` (the doctor's message) and `final_intent`
(the router's prediction), plus the fixed policy and six development examples.
The rules in [the prompt file](prompts/llm_as_judge_prompt.py) require all three:

1. The request uses indirect meaning, shorthand, or incomplete execution details.
2. The predicted intent is indefensible under every reasonable policy reading.
3. The mismatch is a failure to recover that scheduling meaning.

A direct polarity error or a prohibited patient-specific operation is a different
category. Missing dates alone do not make a correctly recognized intent fail.
Human labels are used only for scoring; the six prompt examples are never scored.
The reference notebook's frozen calibration/validation split is reused unchanged.


In [ ]:
rows = load_dataset(LAB / "data", split=SPLIT)
print("Category:", FAILURE_CATEGORY)
print("Question:", FAILURE_QUESTION)
display(pd.DataFrame(CHOICE_CRITERIA.items(), columns=["answer", "meaning"]))
print("Split:", SPLIT, "| cases:", len(rows))
print("State sent to Jev:", request_for(rows[0], "jev")["state"])
# Reference labels are deliberately absent from the model-input preview.
display(pd.DataFrame(rows)[["id", "user_query", "predicted_intent"]].head(3))


## 3. Call Jev and validate its output

Jev answers the binary question with **Choice** options `TRUE` and `FALSE`.
The helper validates the response type, option, probabilities, and native confidence.
Malformed or missing judgments remain unresolved and will be sent to the LLM.
Choice exposes native confidence; confidence is not measured accuracy.

[TypeSafe Choice documentation](https://docs.typesafe.ai/primitives/choice).


In [ ]:
jev = batch(rows, lambda row: client.call(row, "jev"))
example_id = rows[0]["id"]
display(pd.DataFrame([jev[example_id]]).reindex(columns=["label", "confidence", "probabilities", "error"]))

## 4. Apply the confidence gate and LLM fallback

**Jev call → validate output → confidence gate → LLM fallback → final verdict → logging.**

Keep a valid Jev answer at confidence **0.70 or above**, for either TRUE or FALSE.
Below 0.70, or after invalid Jev output, call the LLM with the same evidence and rubric.
If that call fails, leave the final verdict unresolved; do not convert it to FALSE.

For binary Choice, probabilities 0.85 and 0.15 give confidence 0.70. The gate uses
returned confidence, not P(TRUE). This cutoff is fixed, not tuned on these labels.
[Confidence definition](https://docs.typesafe.ai/confidence).


In [ ]:
threshold = 0.70
cascade = run_cascade(rows, client, threshold, jev=jev)
escalated_ids = {row["id"] for row in rows if needs_llm(jev[row["id"]], threshold)}
assert {key for key, value in cascade.items() if value["route"] == "LLM"} == escalated_ids
print(f"LLM fallback used for {len(escalated_ids)}/{len(rows)} cases.")


## 5. Run the independent LLM-only baseline

GPT-4.1-mini receives the same original rules and six worked examples, without Jev's
answer or confidence. It returns a Boolean label and a short explanation. Its prompt
is unchanged from the original LLM notebook; this comparison uses a different model.

Complete the LLM-only run on every case. Exact-request caching reuses the fallback
calls already made above. This benchmark therefore adds LLM calls only for cases
that Jev kept; these extra calls are not part of a standalone cascade's cost.


In [ ]:
llm = batch(rows, lambda row: client.call(row, "llm"))
display(pd.DataFrame([llm[example_id]]).reindex(columns=["label", "reason", "error"]))


## 6. Compare results

Report quality on the **same cases successfully judged by every method**, as in the
reference notebook. Coverage and unresolved counts use all requested cases, so missing
judgments remain visible. FALSE positives and FALSE negatives count resolved judgments.

Check TRUE recall and false alarms alongside agreement and the always-FALSE baseline.
Costs include all attempts, even failed ones. They use reported charges or dated
price estimates. Cascade cost includes Jev on every case and only the fallback LLM
calls; the full experiment runs both judges on every case.


In [ ]:
approaches = {"Always FALSE": always_false(rows), "Jev": jev, "LLM": llm, "Jev + LLM": cascade}
scored_ids = [r["id"] for r in rows if all(resolved(a[r["id"]]) for a in approaches.values())]
comparison = pd.DataFrame(comparison_metrics(rows, approaches))
comparison["cost_per_1000_USD"] = comparison.cost_usd / len(rows) * 1000
comparison["LLM_fraction"] = comparison.llm_routes / len(rows)
comparison.loc[comparison.method.eq("LLM"), "LLM_fraction"] = 1.0
columns = ["method", "scored_n", "coverage", "accuracy", "precision_TRUE", "recall_TRUE", "f1_TRUE",
           "FP", "FN", "unresolved", "LLM_fraction", "cost_per_1000_USD"]
table = comparison[columns].copy()
table.loc[comparison.unknown_cost_attempts.gt(0), "cost_per_1000_USD"] = float("nan")
display(table.round(4))


## 7. Log decisions and inspect disagreements

Inspect the request, prediction, explanation, and provisional reference together.
Within paired successful cases, count LLM fixes versus regressions. Report missing
judgments separately. Confident Jev mistakes bypass the fallback, so review those too.
These practice results do not establish performance on new traffic.

The decision log records the category, question, confidence, routing reason, final
verdict, errors and request hashes. The run manifest records the split, model, prompt
hash and scoring IDs. Credentials are never logged.


In [ ]:
bands = confidence_bands(rows, jev, llm, threshold)
display(pd.DataFrame(bands))

records = []
for row in rows:
    case_id = row["id"]
    j, l, final = jev[case_id], llm[case_id], cascade[case_id]
    gate_reason = ("invalid_jev_output" if j.get("error") else
                   "below_confidence_threshold" if case_id in escalated_ids else
                   "confidence_at_or_above_threshold")
    records.append({
        "ID": case_id, "split": SPLIT, "category": FAILURE_CATEGORY, "question": FAILURE_QUESTION,
        "request": row["user_query"], "predicted_intent": row["predicted_intent"],
        "human": row["human_label"], "included_in_paired_scores": case_id in scored_ids, "Jev": j["label"], "Jev_confidence": j.get("confidence"),
        "threshold": threshold, "gate_reason": gate_reason, "route": final["route"],
        "LLM": l["label"], "LLM_reason": l.get("reason"), "final": final["label"],
        "Jev_error": j.get("error"), "LLM_error": l.get("error"), "final_error": final.get("error"),
        "Jev_request_hash": j["request_hash"], "LLM_request_hash": l["request_hash"],
        "cascade_cost_usd": final["cost_usd"], "unknown_cost_attempts": final["unknown_cost_attempts"],
    })
review = pd.DataFrame(records)
disagreements = review[review.Jev.ne(review.LLM) | review.final.ne(review.human)]
display(disagreements.head(8))
review.to_csv(RUN / "all_results.csv", index=False)
disagreements.to_csv(RUN / "disagreements.csv", index=False)
save_run(client, RUN, manifest, rows, threshold, comparison, bands, split=SPLIT, score_ids=scored_ids)
